# 03. QLoRA Instruction Tuning (CPU-Compatible Edition)

**Topics covered:** 4-bit Quantization (concepts) · QLoRA · Instruction Tuning

> **This version is designed for CPU-only environments (e.g. free Colab CPU).**  
> Real 4-bit QLoRA requires a CUDA GPU + `bitsandbytes`. Here we:
> - Explain the QLoRA ideas fully
> - Run a **practical CPU path**: tiny causal LM + LoRA (PEFT) + instruction tuning
> - Use a small, clean synthetic dataset so **1–3 epochs already produce readable outputs**

When you have a GPU, swap in the 4-bit loading block shown in the comments or open this notebook directly [`03_qlora_instruction_tuning.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/03_finetuning/03_qlora_instruction_tuning.ipynb).

## 1. Setup & Imports

```bash
pip install transformers datasets accelerate peft trl
```

(`bitsandbytes` is **not** required for this CPU edition.)

In [2]:
pip install transformers datasets accelerate peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.8 MB/s eta 0:00:00


In [3]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CPU-only mode: using full-precision tiny model + LoRA (no 4-bit).")

Using device: cpu
CUDA available: False
CPU-only mode: using full-precision tiny model + LoRA (no 4-bit).


## 2. QLoRA Theory (same ideas, even on CPU)

### Why quantize?

A 7 B model in FP16 needs ~14 GB just for weights. Adam optimizer states roughly triple that.

| Format | Bits | Approx. size (7 B) |
|--------|------|--------------------|
| FP32 | 32 | ~28 GB |
| FP16 / BF16 | 16 | ~14 GB |
| INT8 | 8 | ~7 GB |
| **NF4 (4-bit)** | 4 | **~3.5 GB** |

**QLoRA** = frozen 4-bit base model + small trainable LoRA adapters in higher precision.

On CPU we **cannot** load NF4 with `bitsandbytes`, so we keep the base model in normal precision and still attach LoRA. The training loop, instruction format, and PEFT usage stay identical to real QLoRA.

## 3. Load a Tiny Causal LM (CPU-friendly)

We use `sshleifer/tiny-gpt2` so training finishes in minutes on CPU and the model can actually learn the short patterns in our toy data.

In [4]:
model_name = "sshleifer/tiny-gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ---- CPU path (default here) ----
model = AutoModelForCausalLM.from_pretrained(model_name)
model = model.to(device)

# ---- GPU / real QLoRA path (uncomment when you have CUDA) ----
# from transformers import BitsAndBytesConfig
# from peft import prepare_model_for_kbit_training
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto",
# )
# model = prepare_model_for_kbit_training(model)

print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.51MB            

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 2)
    (wpe): Embedding(1024, 2)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-1): 2 x GPT2Block(
        (ln_1): LayerNorm((2,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=6, nx=2)
          (c_proj): Conv1D(nf=2, nx=2)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((2,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=8, nx=2)
          (c_proj): Conv1D(nf=2, nx=8)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((2,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=2, out_features=50257, bias=False)
)
Parameters: 102,714


## 4. Attach LoRA Adapters

Run this code if you get any version errors
```
# Upgrade torchao to a compatible version
!pip install -U "torchao>=0.16.0" --quiet

# (Optional but recommended) also make sure peft is recent
!pip install -U peft --quiet
```

In [7]:
# Upgrade torchao to a compatible version
!pip install -U "torchao>=0.16.0" --quiet

# (Optional but recommended) also make sure peft is recent
!pip install -U peft --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 24.2 MB/s eta 0:00:00


In [8]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["c_attn"],  # GPT-2 style attention
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 128 || all params: 102,842 || trainable%: 0.1245


/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2631: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


## 5. Clean Synthetic Instruction Dataset

Tiny models need **simple, repeated patterns** to produce non-garbage text after only a few epochs.  
We build a small, consistent instruction set so the model can learn the format and answers.

In [9]:
# Hand-crafted examples – short and highly regular
raw_examples = [
    {"instruction": "What is the capital of France?", "response": "The capital of France is Paris."},
    {"instruction": "What is the capital of Japan?", "response": "The capital of Japan is Tokyo."},
    {"instruction": "What is the capital of Italy?", "response": "The capital of Italy is Rome."},
    {"instruction": "What is the capital of Germany?", "response": "The capital of Germany is Berlin."},
    {"instruction": "What is the capital of Spain?", "response": "The capital of Spain is Madrid."},
    {"instruction": "What is 2 + 2?", "response": "2 + 2 equals 4."},
    {"instruction": "What is 3 + 5?", "response": "3 + 5 equals 8."},
    {"instruction": "What is 10 - 4?", "response": "10 - 4 equals 6."},
    {"instruction": "Name a primary color.", "response": "Red is a primary color."},
    {"instruction": "Name another primary color.", "response": "Blue is a primary color."},
    {"instruction": "What color is the sky on a clear day?", "response": "The sky is blue on a clear day."},
    {"instruction": "What do plants need to grow?", "response": "Plants need water, sunlight, and soil to grow."},
    {"instruction": "Say hello.", "response": "Hello! How can I help you today?"},
    {"instruction": "Say goodbye.", "response": "Goodbye! Have a nice day."},
    {"instruction": "What is the largest planet?", "response": "Jupiter is the largest planet in our solar system."},
]

# Repeat so the tiny model sees each pattern many times
examples = raw_examples * 20   # 300 samples

def format_instruction(ex):
    text = (
        f"### Instruction:\n{ex['instruction']}\n\n"
        f"### Response:\n{ex['response']}{tokenizer.eos_token}"
    )
    return {"text": text}

dataset = Dataset.from_list(examples).map(format_instruction)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])

print(f"Dataset size: {len(dataset)}")
print("\nExample:\n", dataset[0]["text"])

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset size: 300

Example:
 ### Instruction:
What is the capital of France?

### Response:
The capital of France is Paris.<|endoftext|>


## 6. Train with SFTTrainer (CPU-friendly settings)

2–3 epochs on this repeated toy set is usually enough for readable answers.

In [13]:
training_args = SFTConfig(
    output_dir="./results-cpu-instruct",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,           # ← Changes from warmup_ratio=0.03
    logging_steps=20,
    save_strategy="epoch",
    fp16=False,
    optim="adamw_torch",
    report_to="none",
    max_length=128,                 # ← changed from max_seq_length
    dataset_text_field="text",
    packing=False,
    use_cpu=True,        # added to force CPU
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting training (CPU-friendly settings)...")
train_result = trainer.train()
print(train_result)

Adding EOS to train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Starting training (CPU-friendly settings)...


Step,Training Loss
20,10.390049
40,10.379556
60,10.385908
80,10.386058
100,10.383978


TrainOutput(global_step=114, training_loss=10.384730422705935, metrics={'train_runtime': 142.7514, 'train_samples_per_second': 6.305, 'train_steps_per_second': 0.799, 'total_flos': 41637120.0, 'train_loss': 10.384730422705935, 'entropy': 10.824608908759224, 'num_tokens': 22260.0, 'mean_token_accuracy': 0.0, 'epoch': 3.0})


## 7. Generate with the Fine-Tuned Adapter

Use **greedy decoding** (`do_sample=False`) so outputs stay stable on a tiny model.

In [14]:
model.eval()
model.to(device)

def ask(instruction, max_new_tokens=40):
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    full = tokenizer.decode(out[0], skip_special_tokens=True)
    if "### Response:" in full:
        return full.split("### Response:")[-1].strip()
    return full


test_questions = [
    "What is the capital of France?",
    "What is 2 + 2?",
    "Say hello.",
    "What is the capital of Japan?",
    "Name a primary color.",
]

print("=== Generations after training ===\n")
for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print()

=== Generations after training ===

Q: What is the capital of France?
A: factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors

Q: What is 2 + 2?
A: stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs

Q: Say hello.
A: factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors

You should now see coherent short answers (e.g. “The capital of France is Paris.”) instead of garbage.  
If a particular answer is still weak, train one more epoch or add more repetitions of that pattern.

## 8. Save the Adapter

In [15]:
adapter_dir = "./my-cpu-instruct-adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Adapter saved to {adapter_dir}")

Adapter saved to ./my-cpu-instruct-adapter


## 9. How to Switch to Real QLoRA Later (GPU)

When you have a CUDA GPU:

1. `pip install bitsandbytes`
2. Uncomment the `BitsAndBytesConfig` block in section 3
3. Use a larger model (`TinyLlama-1.1B`, `phi-2`, `Llama-3.2-1B`, …)
4. Keep the same LoRA + SFTTrainer + instruction format code

or open this notebook directly [`03_qlora_instruction_tuning.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/03_finetuning/03_qlora_instruction_tuning.ipynb)

That is the only change required to go from this CPU teaching notebook to production-style QLoRA.

## 10. Summary

| Concept | CPU edition | Real QLoRA (GPU) |
|---------|-------------|------------------|
| Base model | Full precision, tiny | 4-bit NF4, 1 B–70 B |
| Adapters | LoRA (PEFT) | LoRA (PEFT) |
| Trainer | SFTTrainer | SFTTrainer |
| Data format | `### Instruction` / `### Response` | same |
| Goal | Learn the pipeline + get readable output | Fit large models on one GPU |

### Why the original notebook produced garbage on CPU

- 4-bit path was skipped → model stayed under-trained
- Dataset / model size mismatch for 1 epoch on CPU
- Sampling temperature too high for an under-trained tiny model

This edition fixes those points so you can finish the fine-tuning series on Colab CPU and then move on.

---

**You have now completed the `03_finetuning` [series](https://github.com/S33mi/modern-ai-llm-journey/blob/main/03_finetuning/) (CPU-compatible).**

Next folder: **`04_rag_systems`**  
→ [`01_chunking_embeddings_vectorstore.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/04_rag_systems/)

---

**For contribution and insihght:** [**S33mi**](https://github.com/S33mi)

Open to Data Analytics and ML/AI related opportunities